# Inferencia para Transformers

In [1]:
!pip install "transformers==4.46.1" "trl==0.12.0"
!pip install psutil # para medir la RAM

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.5 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import psutil
import torch
import time
import os

### Configuración global (para el resto de procesos de inferencia será parecido)

In [3]:
print("Cargando dataset Dolly 15k...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
SAMPLE_SIZE = 75
test_subset = dataset.shuffle(seed=42).select(range(SAMPLE_SIZE)) # se usa la misma semilla en todos para que la comparación sea justa

# --- PARÁMETROS ---
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PROMPTS = [f"Instruction: {row['instruction']}\nResponse:" for row in test_subset]

# Configuración de generación idéntica para todos
GEN_CONFIG = {
    "max_new_tokens": 128,
    "temperature": 0.7,
    "top_k": 50,
    "do_sample": True
}

# Tabla para guardar resultados
results = []

# Determine if CUDA is available, otherwise use CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
    print("CUDA está disponible. Usando GPU.")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32 # float16 is usually not optimized for CPU
    print("CUDA no está disponible. Usando CPU.")

def get_vram_usage():
    """Devuelve la memoria RAM/VRAM usada en GB"""
    if DEVICE == "cuda":
        return torch.cuda.max_memory_allocated() / (1024**3)
    else:
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / (1024**3)

def clear_cache():
    """Limpia la memoria GPU entre pruebas, o no hace nada si es CPU"""
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats() # Only call if CUDA is available
    import gc
    gc.collect()

print(f"Modelo: {MODEL_ID}")
print(f"Número de pruebas (Prompts): {len(PROMPTS)}")
print(f"Ejemplo de prompt:\n{PROMPTS[0][:100]}...")

Cargando dataset Dolly 15k...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

CUDA está disponible. Usando GPU.
Modelo: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Número de pruebas (Prompts): 75
Ejemplo de prompt:
Instruction: Who were the children of the legendary Garth Greenhand, the High King of the First Men ...


# 1. Hugging Face



In [4]:
def benchmark_hf():
    print("--- Iniciando Benchmark: Transformers ---")
    clear_cache()

    # Cargar modelo
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE, # Usar DTYPE determinado por la disponibilidad de GPU
        device_map=DEVICE if DEVICE == "cuda" else None # 'None' para CPU deja que PyTorch decida
    ).to(DEVICE)

    # Warmup
    inputs = tokenizer("Test", return_tensors="pt").to(DEVICE)
    model.generate(**inputs, max_new_tokens=5)
    clear_cache() # Resetear contadores de memoria post-carga

    latencies = []
    tokens_per_sec = []

    start_vram = get_vram_usage()

    # Loop
    for prompt in tqdm(PROMPTS, desc="Transformers"):
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        input_len = inputs.input_ids.shape[1]

        start_time = time.time()
        with torch.no_grad():
            output = model.generate(**inputs, **GEN_CONFIG)
        end_time = time.time()

        gen_tokens = output[0].shape[0] - input_len
        duration = end_time - start_time

        latencies.append(duration)
        tokens_per_sec.append(gen_tokens / duration)

    peak_vram = get_vram_usage()

    # Guardar datos
    results.append({
        "Framework": "Transformers",
        "Avg Latency (s)": np.mean(latencies),
        "Throughput (tok/s)": np.mean(tokens_per_sec),
        "Peak RAM / VRAM (GB)": peak_vram # Esto será 0 si no hay GPU
    })

    # Limpieza final
    del model, tokenizer
    clear_cache()
    print("Transformers completado y memoria liberada.")

benchmark_hf()

--- Iniciando Benchmark: Transformers ---


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Transformers: 100%|██████████| 75/75 [02:55<00:00,  2.34s/it]


Transformers completado y memoria liberada.


In [5]:
results

[{'Framework': 'Transformers',
  'Avg Latency (s)': np.float64(2.3393377590179445),
  'Throughput (tok/s)': np.float64(38.8815723823371),
  'Peak RAM / VRAM (GB)': 2.0674009323120117}]